## RAG 시스템 배포

**고객 상담 RAG 에이전트**를 Azure 클라우드에 배포하기

---

### 1. RAG 시스템 아키텍처


### 주요 컴포넌트
1.  **Azure Container App (Compute)**: 
    -   파이썬 코드(`main.py`)가 실행되는 곳
2.  **Azure AI Search (Knowledge)**:
    -   약관 데이터가 저장된 곳으로 단순 키워드 검색뿐만 아니라 의미 기반(Vector) 검색을 수행
3.  **Azure OpenAI (Engine)**:
    -   검색된 내용을 바탕으로 사람처럼 결과를 출력합니다.

## 2. 코드 상세 분석


In [15]:
!pip freeze | grep -E "langchain|langgraph|azure-search-documents"

azure-search-documents==11.6.0
langchain==1.2.10
langchain-anthropic==1.3.4
langchain-classic==1.0.1
langchain-community==0.4.1
langchain-core==1.2.15
langchain-google-genai==4.2.1
langchain-mcp-adapters==0.2.1
langchain-openai==1.1.10
langchain-tavily==0.2.17
langchain-text-splitters==1.1.1
langgraph==1.0.9
langgraph-checkpoint==4.0.0
langgraph-prebuilt==1.0.8
langgraph-sdk==0.3.9
langgraph-supervisor==0.0.31
langgraph_checkpoint_cosmosdb==0.2.5
Note: you may need to restart the kernel to use updated packages.


In [14]:
from langchain.agents import create_agent
help(create_agent)


import inspect
from langchain.agents import create_agent
print(inspect.signature(create_agent))

Help on function create_agent in module langchain.agents.factory:

create_agent(model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache[Any] | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]]'
    Creates an agent graph that calls tools in a loop until a stopping condition is met.

    For more details on using `cre

In [1]:
!echo ${USER}

azureuser


In [2]:
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from dotenv import load_dotenv

# .env는 상위 폴더에 위치
load_dotenv("../../env")

app = FastAPI(title="Azure AI RAG Service")

# 환경 변수 (.env 기준)
endpoint = os.getenv("END_POINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model_name = os.getenv("MODEL_NAME")
search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
search_key = os.getenv("AZURE_SEARCH_KEY")
index_name = "telecom-terms-index"

if endpoint:
    endpoint = endpoint.rstrip("/")
    if ".openai.azure.com" in endpoint and model_name and "/openai/deployments/" not in endpoint:
        endpoint = f"{endpoint}/openai/deployments/{model_name}"
    elif "/chat/completions" in endpoint:
        endpoint = endpoint.split("/chat/completions")[0].split("?")[0]

if search_endpoint and "/" in search_endpoint.replace("://", ""):
    base_url = search_endpoint.split("://")[-1].split("/")[0]
    protocol = search_endpoint.split("://")[0]
    search_endpoint = f"{protocol}://{base_url}"


### 2.1. 클라이언트 초기화 (Setup)
가장 먼저 `SearchClient`(검색용)와 `ChatCompletionsClient`(생성용)를 설정합니다.
환경 변수에서 Endpoint와 Key를 가져와서 연결합니다.

In [3]:
def get_search_client():
    credential = AzureKeyCredential(search_key) if search_key else DefaultAzureCredential()
    return SearchClient(endpoint=search_endpoint, index_name=index_name, credential=credential)

def get_inference_client():
    # 키 우선, 없으면 Managed Identity 등 DefaultAzureCredential 사용
    if api_key:
        credential = AzureKeyCredential(api_key)
    else:
        credential = DefaultAzureCredential()
    return ChatCompletionsClient(endpoint=endpoint, credential=credential)


In [4]:
# az containerapp logs show   --name rag-agent-api   --resource-group rg-KT-new-Foundry   --follow

### 2.2. 문서 검색 (Retrieve)
사용자의 질문을 받아 Azure AI Search에서 가장 관련성 높은 문서 3개를 찾아옵니다.
이때 `page` 필드도 함께 가져와서 나중에 출처를 표시할 때 사용합니다.

In [ ]:
def retrieve_documents(query: str):
    client = get_search_client()
    # 상위 3개 문서 검색
    results = client.search(search_text=query, top=3, select=["content", "page"])
    
    context = ""
    sources = set()
    for result in results:
        page_num = result.get("page")
        # 검색된 내용을 가운데 --- 만 넣어서 하나의 긴 텍스트(context)로 합침
        context += f"---\n[Page {page_num}] {result['content']}\n"
        if page_num:
            sources.add(str(page_num))
        
    return context, sorted(list(sources))

### 2.3 답변 생성 (Generate)

In [6]:
class ChatRequest(BaseModel):
    query: str

@app.post("/chat")
def chat_endpoint(request: ChatRequest):
    # 1. 검색 수행
    context, sources = retrieve_documents(request.query)
    
    # 2. 프롬프트 구성
    system_prompt = """당신은 통신사 고객 상담을 담당하는 친절하고 전문적인 AI 상담원입니다.
    제공된 [약관 내용]을 바탕으로 고객의 질문에 정확하게 답변해 주세요..."""
    
    user_message = f"[약관 내용]\n{context}\n\n[고객 질문]\n{request.query}"
    
    # 3. GPT 호출
    client = get_inference_client()
    response = client.complete(
        messages=[
            SystemMessage(content=system_prompt),
            UserMessage(content=user_message),
        ]
    )
    
    return {"response": response.choices[0].message.content, "sources": sources}

## 3. 배포 설정 (Dockerfile)

이 코드를 클라우드에서 실행하려면 **컨테이너(Container)**로 만들어야 합니다.
`Dockerfile`은 그 설계도입니다.

## 4. 배포 시 주의사항: 환경 변수 (Environment Variables)

RAG 시스템은 연결해야 할 서비스가 많아서 환경 변수가 더 많습니다.


이 값들은 배포 명령어(`az containerapp up`) 실행 시 `--env-vars` 옵션으로 전달해야 합니다.


- `END_POINT` : Azure OpenAI 리소스 주소 (04번과 동일) 
- `AZURE_OPENAI_API_KEY` : OpenAI API 키 
- `MODEL_NAME` : 배포 이름 (예: gpt-5-mini) 
- `AZURE_SEARCH_ENDPOINT` : Azure AI Search 주소 
- `AZURE_SEARCH_KEY` : Search 인증키 
- `INDEX_NAME` : (선택) 검색 인덱스 이름, 기본값 telecom-terms-index 


## 5. 배포 명령어 실행

이제 실제로 배포를 진행합니다. (약 3~5분 소요)

```bash
az containerapp up \
  --name rag-agent-api \              # 앱 이름
  --resource-group rg-kt-project \    # 리소스 그룹
  --source . \                        # 현재 폴더 소스 사용
  --ingress external \                # 외부 접속 허용
  --target-port 8000 \                # FastAPI 포트
  --env-vars ...                      # 위에서 설명한 4개의 변수 필수!
```

In [7]:
!sh deploy_cmd.sh

deploy_cmd.sh: 1: source: not found
Using resource group 'rg-KT-new-Foundry'
Using ContainerAppEnvironment 'foundry-basic-api-2-env' in resource group rg-KT-new-Foundry
2026/02/27 02:17:49 Downloading source code...
2026/02/27 02:17:50 Finished downloading source code
2026/02/27 02:17:50 Using acb_vol_da5c2191-980a-4b07-a33a-15384a1038dc as the home volume
2026/02/27 02:17:50 Setting up Docker configuration...
2026/02/27 02:17:50 Successfully set up Docker configuration
2026/02/27 02:17:50 Logging in to registry: cad686be8f39acr.azurecr.io
2026/02/27 02:17:51 Successfully logged into cad686be8f39acr.azurecr.io
2026/02/27 02:17:51 Executing step ID: build. Timeout(sec): 28800, Working directory: '', Network: ''
2026/02/27 02:17:51 Scanning for dependencies...
2026/02/27 02:17:51 Successfully scanned dependencies
2026/02/27 02:17:51 Launching container with name: build
Sending build context to Docker daemon  166.9kB
Step 1/7 : FROM python:3.12-slim
3.12-slim: Pulling from library/python


In [8]:
# %pip install gradio

In [9]:
https://rag-agent-api.delightfulpond-e9f0d359.westus2.azurecontainerapps.io/docs#/default/chat_chat_post

## 7. 인터랙티브 챗봇 인터페이스

배포된 RAG API를 사용하는 Gradio 챗봇 인터페이스입니다.

ml studio가 아니라 로컬에서 실행하는 분들은 노트북 셀에서 직접 대화를 나눌 수 있습니다.

In [ ]:
# 임시로 필요한 패키지 설치 (노트북에서만 사용)
# %pip install -q gradio requests

In [ ]:
# gradio_chatbot.py에서 챗봇 인터페이스 불러오기
from gradio_chatbot import create_chatbot_interface

# 챗봇 인터페이스 생성 및 실행 (노트북에 인라인으로 표시)
demo = create_chatbot_interface()
demo.launch(inline=True, share=False)

## 6. 테스트 및 결과 확인

배포가 성공하면 `test_rag_remote.py`를 통해 테스트를 진행합니다.

1.  사용자가 질문("약정 기간은?")을 입력
2.  배포된 API 서버(`/chat`)로 전송
3.  서버가 Azure Search에서 관련 문서를 찾음
4.  GPT가 답변을 생성하고 출처를 붙임
5.  최종 응답을 받아 출력


In [ ]:
!python3 test_rag_remote.py